Installing libraries

In [1]:
import os
import json
import requests
import time
import datetime as dt

from zoneinfo import ZoneInfo
from email.utils import parsedate_to_datetime

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [3]:
# Load environment variables
load_dotenv(override= True)

openai_api_key = os.getenv("OPENAI_API_KEY")
twitterapi_io_key = os.getenv("TWITTERAPI_IO_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins with {openai_api_key[:7]}.......")
else:
    print("OpenAI API Key is not set")

if twitterapi_io_key:
    print(f"Twitter API Key exists and begins with {twitterapi_io_key[:8]}.....")
else:
    print(f"Twitter API is not set.")

openai_client = OpenAI()

twitter_base_url = "https://api.twitterapi.io"
TZ = ZoneInfo("America/Chicago")

print("Setup complete")

OpenAI API Key exists and begins with sk-proj.......
Twitter API Key exists and begins with fskngnfd.....
Setup complete


Fetch tweets from twitterapi.io  
Free-tier: 1 reequest every 5 seconds, includes 429 backoff handling


In [4]:
def fetch_users_last_tweets(username: str, limit: int= 80) -> list[dict]:
    """
    Fetch recent tweets for a username via twitterapi.io
    """

    url = f"{twitter_base_url}/twitter/user/last_tweets"
    headers= {"X-API_KEY: twitter_api_io_key"}
    params = {'username': username, 'count': limit, 'limit': limit}

    r = requests.get(url= url, headers= headers, params= params)

    # Retry once if rate-limited
    if r.status_code == 429:
        time.sleep(6)
        r = requests.get(url= url, headers= headers, params= params)

    if not r.ok:
        raise RuntimeError(f"Twitter API error: {r.status_code}: {r.text[:800]}")
    
    data = r.json()

    # Handles various response shapes from the API
    tweets = {
        data.get("data", {}).get("tweets")
        or data.get("tweets")
        or data.get("data", {}).get("items")
        or data.get("data")
        or []
    }

    if isinstance(tweets, dict):
        tweets = tweets.get("tweets") or tweets.get("items") or []

    return tweets[:limit]


In [6]:
def _parse_created_at(tweet: dict) -> dt.datetime | None:
    """
    Parse tweet timestamp and convert to local timezone.
    """

    s = tweet.get("CreatedAt") or tweet.get("created_at")

    if not s:
        return None
    
    try:
        return parsedate_to_datetime(s).astimezone(TZ)
    except Exception:
        return None
    

def is_retweet(tweet: dict) -> bool:
    return tweet.get("retweeted_tweet") is not None

def tweet_text(tweet: dict) -> str:
    return (tweet.get("text") or "")

def tweet_url(tweet: dict) -> str:
    return (tweet.get("url") or tweet.get("twitterUrl") or "").strip()


def split_today_vs_yesterday(tweets: list[dict], include_retweets: bool = True) -> tuple[list[dict], list[dict]]:
    """
    Seperate tweets into today vs yesterday's buckets
    """

    now = dt.datetime.now(TZ)
    today, yesterday = now.date(), (now - dt.timedelta(days=1)).date()

    todays, yesterdays = [], []

    for tw in tweets:
        if not include_retweets and is_retweet(tw):
            continue

        t = _parse_created_at(tw)

        if not t:
            continue

        if t.date() == today:
            todays.append(tw)
        elif t.date() == yesterday:
            yesterdays.append(tw)

    sort_key = lambda tw: (_parse_created_at(tw) or dt.datetime.min.replace(tzinfo=TZ)).timestamp()

    return sorted(todays, key= sort_key), sorted(yesterdays, key= sort_key)       


In [7]:
def format_tweets_for_LLM(tweets: list[dict], max_items: int= 50) -> str:
    """
    Format tweet list into a compact text format for the LLM.
    """

    if not tweets:
        return ("No tweets found.")
    
    lines = []

    for tw in tweets:
        t = _parse_created_at(tw)
        ts = t.strftime("%Y-%m-%d %H:%M %Z") if t else "Unknown Time"
        prefix = "RT: " if is_retweet(tw) else ""
        text = tweet_text(tw)
        url = tweet_url(tw)
        lines.append(f"- [{ts}] {prefix}{text} ({url})").strip()

    return "\n".join(lines)

In [8]:
system_prompt = """
You are an sassy snarky assistant intelligent assistant.
You will be given two sets of tweets from the same account.
(1) today's tweet
(2) yesterday's tweet

Product a crisp daily brief with these sections: 

1) What changed vs yesterday?
2) What's new product / market signal?
3) Anything controversial/risky?
4) Actionable items:
    - Investigate (max 3)
    - Monitor (max 3)
    - Ignore (max 3)

Rules: 
- Ground everything in the provided tweets.
- If today's tweets are empty, say so and base "changed vs yesterday" on that.
- Output markdown only (no code block)
""".strip()

In [9]:
def summarize_daily(username: str= "elonmusk", include_retweets: bool = True) -> str:
    """
    Fetch tweets and generate AI-powered daily summary.
    """

    raw = fetch_users_last_tweets(username, 50)

    todays, yesterdays = split_today_vs_yesterday(raw)

    todays_block = format_tweets_for_LLM(todays)
    yesterdays_block = format_tweets_for_LLM(yesterdays)

    user_prompt = f"""
Here are the details of the users twitter account.
Please provide a detailed summary.

Account : @{username}
Timezone: "America/Chicago"
Include retweets: {include_retweets}

Today's tweets:
{todays_block}

yesterday's tweets:
{yesterdays_block}
""".strip()
    
    response = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [10]:
def display_summary(username: str = "elonmusk", include_retweets: bool= True):
    """
    Display the summary as formatted Markdown
    """

    md = summarize_daily(username, include_retweets)
    display(Markdown(md))

In [ ]:
display_summary("iamsrk", include_retweets= True)